In [ ]:
# LLM-as-a-judge.ipynb
%pip install -q transformers accelerate bitsandbytes torch sentence-transformers faiss-cpu pandas numpy seaborn
import os
import json
import glob
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.notebook import tqdm

# --- Настройка путей ---
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
RESULTS_DIR = "../artifacts/results/LLM-as-a-judge"
OUTPUT_DIR = "../artifacts/results/LLM-as-a-judge"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- ЗАГРУЗКА МОДЕЛИ-СУДЬИ ---
print(f"Загрузка модели судьи: {JUDGE_MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID, trust_remote_code=True)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
)
judge_model.eval()
print("Модель судьи загружена.")

In [ ]:
def evaluate_recommendations(user_profile, recommendations, raw_output_model):
    """
    Формирует промпт и получает оценку от модели-судьи.
    """

    # Формируем краткое описание профиля для контекста
    fav_tags_str = ", ".join([t for t, _ in user_profile.get('favorite_tags', [])[:5]])
    liked_books_str = "\n".join([f"- {b['title']} (Rating: {b['rating']})" for b in user_profile.get('liked_books', [])[:3]])

    # Формируем список рекомендованных книг для оценки
    recs_context = ""
    for i, rec in enumerate(recommendations):
        recs_context += f"""
Book {i+1}:
Title: {rec['title']}
Authors: {rec['authors']}
Reason given by Recommender Model: {rec.get('reason', 'No reason provided')}
"""

    system_prompt = """
You are an expert Book Recommendation Judge. Your task is to evaluate the quality of book recommendations made by an AI system for a specific user.

You will be provided with:
1. User Profile: Their favorite tags, liked books, and average rating behavior.
2. Recommendations: A list of books recommended by the AI, along with the AI's reasoning.

Evaluation Criteria (Score 1-10):
- Relevance (4 pts): Do the books match the user's favorite tags and authors?
- Diversity (2 pts): Are the recommendations varied enough (not just 5 copies of the same book)?
- Reasoning Quality (2 pts): Is the AI's explanation logical and convincing?
- Novelty/Appropriateness (2 pts): Are the books appropriate for the user's taste level?

Scoring Guide:
1-3: Poor. Irrelevant, wrong genre, or bad reasoning.
4-6: Average. Some relevance, but generic or weak reasoning.
7-8: Good. Relevant, good reasoning, matches taste.
9-10: Excellent. Perfect match, insightful reasoning, discovers hidden gems.

Output Format:
You MUST output a valid JSON object with the following structure:
{
  "evaluations": [
    {
      "book_index": 1,
      "score": 8,
      "explanation": "Short explanation why this score was given."
    },
    ...
  ],
  "overall_model_score": 8.5,
  "final_verdict": "Brief summary of the model's performance."
}
Do not output any text outside the JSON block.
"""

    user_prompt = f"""
### User Profile
- Favorite Tags: {fav_tags_str}
- Liked Books:
{liked_books_str}
- Avg Rating: {user_profile.get('mean_rating', 'N/A')}

### Recommendations to Evaluate
{recs_context}

### Task
Evaluate each recommendation and provide an overall score for the AI model.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Парсинг JSON
    try:
        # Очищаем от markdown блоков ```json ... ``` если они есть
        clean_json = response_text.replace("```json", "").replace("```", "").strip()
        result = json.loads(clean_json)
        return result
    except Exception as e:
        print(f"Error parsing JSON from Judge: {e}")
        print(f"Raw output: {response_text[:200]}")
        return None

In [ ]:
# Получаем список всех JSON файлов с рекомендациями
json_files = glob.glob(os.path.join(RESULTS_DIR, "*.json"))
print(f"Найдено файлов для оценки: {len(json_files)}")

all_evaluations = []

for file_path in tqdm(json_files, desc="Evaluating"):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        user_profile = data.get('user_profile_snapshot', {})
        recommendations = data.get('parsed_recommendations', [])

        if not recommendations:
            continue

        # Запуск судьи
        judge_result = evaluate_recommendations(user_profile, recommendations, data.get('raw_model_output', ''))

        if judge_result:
            # Добавляем мета-информацию
            judge_result['user_id'] = data['metadata']['user_id']
            judge_result['model_id'] = data['metadata']['model_id']
            judge_result['source_file'] = os.path.basename(file_path)

            all_evaluations.append(judge_result)

            # Сохраняем промежуточный результат, чтобы не потерять данные при краше
            out_filename = os.path.join(OUTPUT_DIR, f"eval_{os.path.basename(file_path)}")
            with open(out_filename, 'w', encoding='utf-8') as f:
                json.dump(judge_result, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

print(f"Оценка завершена. Обработано: {len(all_evaluations)} пользователей.")

In [ ]:
# Преобразуем в DataFrame для анализа
df_evals = []
for eval_item in all_evaluations:
    user_id = eval_item['user_id']
    overall_score = eval_item.get('overall_model_score', 0)

    for book_eval in eval_item.get('evaluations', []):
        df_evals.append({
            'user_id': user_id,
            'book_index': book_eval['book_index'],
            'book_score': book_eval['score'],
            'explanation': book_eval['explanation'],
            'overall_model_score': overall_score,
            'verdict': eval_item.get('final_verdict', '')
        })

df_results = pd.DataFrame(df_evals)

if not df_results.empty:
    # Средняя оценка модели по всем пользователям
    avg_model_score = df_results['overall_model_score'].mean()
    print(f"\nИТОГОВАЯ ОЦЕНКА МОДЕЛИ: {avg_model_score:.2f} / 10.0")

    # Распределение оценок книг
    avg_book_score = df_results['book_score'].mean()
    print(f"Средняя оценка рекомендуемых книг: {avg_book_score:.2f} / 10.0")

    # 3. Примеры хороших и плохих оценок
    print("\n--- Пример отличной рекомендации (Score 9-10) ---")
    best_rec = df_results[df_results['book_score'] > 8].iloc[0] if not df_results[df_results['book_score'] > 8].empty else None
    if best_rec is not None:
        print(f"User: {best_rec['user_id']}, Explanation: {best_rec['explanation']}")

    print("\n--- Пример слабой рекомендации (Score < 4) ---")
    worst_rec = df_results[df_results['book_score'] < 4].iloc[0] if not df_results[df_results['book_score'] < 4].empty else None
    if worst_rec is not None:
        print(f"User: {worst_rec['user_id']}, Explanation: {worst_rec['explanation']}")

    # Сохранение итогового отчета
    df_results.to_csv(os.path.join(OUTPUT_DIR, "final_evaluation_report.csv"), index=False)
    print(f"\nОтчет сохранен в: {os.path.join(OUTPUT_DIR, 'final_evaluation_report.csv')}")
else:
    print("Нет данных для анализа.")